In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn pandas gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import evaluate

ModuleNotFoundError: No module named 'evaluate'

In [ ]:
import zipfile

with zipfile.ZipFile("/content/archive (1).zip", 'r') as zip_ref:
    zip_ref.extractall("/content/")

print("Zip açıldı")

Zip açıldı


In [ ]:
import os

os.listdir("/content/")

['.config', '42bin_haber', 'archive (1).zip', 'sample_data']

In [ ]:
import os

os.listdir("/content/42bin_haber")

['okubeni.doc', 'news']

In [ ]:
import os
import pandas as pd

base_path = "/content/42bin_haber/news"
data = []

for label in os.listdir(base_path):
    label_path = os.path.join(base_path, label)

    if os.path.isdir(label_path):
        for file_name in os.listdir(label_path):
            file_path = os.path.join(label_path, file_name)

            if file_name.endswith(".txt"):
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read().strip()

                data.append({
                    "text": text,
                    "label": label
                })

df = pd.DataFrame(data)

print("Veri boyutu:", df.shape)
df.head()

Veri boyutu: (41992, 2)


,text,label
0,Alacakaranlığı ne kadar tanıyorsunuz?\nAlacaka...,kultur-sanat
1,Hayal gücünü uçur!\nÇocukların hayal dünyasını...,kultur-sanat
2,"Goya ""protestolarla"" sahiplerini buldu\nİspany...",kultur-sanat
3,Jazz In Wonderland\nAlt Caz\n10.11.2012\n22:30...,kultur-sanat
4,"Gebelere Balon\n""Tuhaf Alışkanlıklar Kitabı"", ...",kultur-sanat


In [ ]:
df["label"].value_counts()

,count
label,
spor,9997
genel,6673
guncel,5847
dunya,3724
ekonomi,3265
magazin,2792
planet,1953
turkiye,1939
siyaset,1849


In [ ]:
df = df.dropna(subset=["text", "label"])
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df = df[df["text"] != ""]
df = df[df["label"] != ""]
df = df.drop_duplicates()

print("Temizlik sonrası boyut:", df.shape)

Temizlik sonrası boyut: (40409, 2)


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

print("Sınıflar:", label_encoder.classes_)
df.head()

Sınıflar: ['dunya' 'ekonomi' 'genel' 'guncel' 'kultur-sanat' 'magazin' 'planet'
 'saglik' 'siyaset' 'spor' 'teknoloji' 'turkiye' 'yasam']


,text,label,label_id
0,Alacakaranlığı ne kadar tanıyorsunuz?\nAlacaka...,kultur-sanat,4
1,Hayal gücünü uçur!\nÇocukların hayal dünyasını...,kultur-sanat,4
2,"Goya ""protestolarla"" sahiplerini buldu\nİspany...",kultur-sanat,4
3,Jazz In Wonderland\nAlt Caz\n10.11.2012\n22:30...,kultur-sanat,4
4,"Gebelere Balon\n""Tuhaf Alışkanlıklar Kitabı"", ...",kultur-sanat,4


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["label_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label_id"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

NameError: name 'df' is not defined

In [ ]:
train_df = train_df[["text", "label_id"]].rename(columns={"label_id": "label"})
val_df = val_df[["text", "label_id"]].rename(columns={"label_id": "label"})
test_df = test_df[["text", "label_id"]].rename(columns={"label_id": "label"})

print(train_df.head())
print(val_df.head())
print(test_df.head())

                                                    text  label
4173   Yaşı küçük kızlara tecavüz iddiasına 25 gözalt...      3
3021   Şapka kanununa elveda\nSivil anayasada Genelku...      3
36514  Yaşanabilir şehrin anahtarı teknoloji\nBTK Baş...     10
33761  YTÜ'de beton kalıbı çöktü\nYıldız Teknik Ünive...      2
27262  Fiyat artışında lider Antalya\nAntalya, bir ön...      1
                                                    text  label
24850  Batalla'dan çarpıcı açıklamalar\nBursaspor'un ...      9
19500  'Fenerbahçe Meireles'i yollamalı'\nGenel sekre...      9
16435  Barcelona arayı açıyor!\nİspanya 1. Futbol Lig...      9
9553   PKKnın elindeki rehineler getiriliyor\nÇeşitl...      8
26416  i-am the bank\n2si Türk 3ü İngiliz 5 ortağ...      1
                                                    text  label
36937  "Vatandaş kanun yapımına tek tıkla katılabilir...     10
22975  Dallas'ı Durant yıktı\nNBA'de Dallas Mavericks...      9
12181  Devlet televizyonu numarayı yanlı

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 28286
})
Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 6061
})
Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 6062
})


In [ ]:
model_name = "dbmdz/bert-base-turkish-cased"

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizer yüklendi.")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer yüklendi.


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(train_dataset)

Map:   0%|          | 0/28286 [00:00<?, ? examples/s]

Map:   0%|          | 0/6061 [00:00<?, ? examples/s]

Map:   0%|          | 0/6062 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 28286
})


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

NameError: name 'train_df' is not defined

In [ ]:
import os
import pandas as pd

base_path = "/content/42bin_haber/news"
data = []

for label in os.listdir(base_path):
    label_path = os.path.join(base_path, label)

    if os.path.isdir(label_path):
        for file_name in os.listdir(label_path):
            file_path = os.path.join(label_path, file_name)

            if file_name.endswith(".txt"):
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read().strip()

                data.append({
                    "text": text,
                    "label": label
                })

df = pd.DataFrame(data)

print(df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/42bin_haber/news'

In [ ]:
import zipfile
import os

with zipfile.ZipFile("/content/archive (1).zip", "r") as zip_ref:
    zip_ref.extractall("/content/")

print(os.listdir("/content/"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/archive (1).zip'

In [ ]:
import zipfile
import os

with zipfile.ZipFile("/content/sample_data/archive (1).zip", "r") as zip_ref:
    zip_ref.extractall("/content/")

print(os.listdir("/content/"))

['.config', '42bin_haber', 'sample_data']


In [ ]:
import os
import pandas as pd

base_path = "/content/42bin_haber/news"
data = []

for label in os.listdir(base_path):
    label_path = os.path.join(base_path, label)

    if os.path.isdir(label_path):
        for file_name in os.listdir(label_path):
            file_path = os.path.join(label_path, file_name)

            if file_name.endswith(".txt"):
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read().strip()

                data.append({
                    "text": text,
                    "label": label
                })

df = pd.DataFrame(data)

print(df.shape)
df.head()

(41992, 2)


,text,label
0,"20 kuruş ikramiye AHİMye gidiyor\nSGKnın, 10...",ekonomi
1,Borusan'ın kurucusu Asım Kocabıyık hayatını ka...,ekonomi
2,Sigortacı 'zarar etti' fatura vatandaşa çıktı\...,ekonomi
3,MarketWatch - 5 Aralık\n-ABD'de 118 bin kişi i...,ekonomi
4,İsrail'in doğalgazı Türkiye'ye geliyor\n \n İs...,ekonomi


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df["label_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label_id"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (29394, 3)
Validation: (6299, 3)
Test: (6299, 3)


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label', 'label_id', '__index_level_0__'],
    num_rows: 29394
})
Dataset({
    features: ['text', 'label', 'label_id', '__index_level_0__'],
    num_rows: 6299
})
Dataset({
    features: ['text', 'label', 'label_id', '__index_level_0__'],
    num_rows: 6299
})


In [ ]:
from transformers import AutoTokenizer

model_name = "dbmdz/bert-base-turkish-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer yüklendi.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer yüklendi.


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(train_dataset)

Map:   0%|          | 0/29394 [00:00<?, ? examples/s]

Map:   0%|          | 0/6299 [00:00<?, ? examples/s]

Map:   0%|          | 0/6299 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'label_id', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 29394
})


In [ ]:
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label_id"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label_id"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label_id"])

In [ ]:
train_df = train_df.rename(columns={"label_id": "label"})
val_df = val_df.rename(columns={"label_id": "label"})
test_df = test_df.rename(columns={"label_id": "label"})

In [ ]:
train_df.head()

,text,label,label
19049,Savaş Ay İstanbul'un kıraathanelerini anlattı\...,kultur-sanat,4
30737,Fenerbahçe Bienvenu'yu borsaya bildirdi\nFener...,spor,9
31646,Elano'dan akıllara zarar gol\n \n Brezilya tak...,spor,9
7432,Müslüm Gürses sahada unutulmadı\nMüslüm Gürses...,guncel,3
27525,Kartal görünümlü Kanarya\nF.Bahçe kadrosunda b...,spor,9


In [ ]:
# df ana verisinden temiz şekilde tekrar split yapıyoruz
from sklearn.model_selection import train_test_split

model_df = df[["text", "label_id"]].rename(columns={"label_id": "label"})

train_df, temp_df = train_test_split(
    model_df,
    test_size=0.3,
    random_state=42,
    stratify=model_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (29394, 2)
Validation: (6299, 2)
Test: (6299, 2)


,text,label
19049,Savaş Ay İstanbul'un kıraathanelerini anlattı\...,4
30737,Fenerbahçe Bienvenu'yu borsaya bildirdi\nFener...,9
31646,Elano'dan akıllara zarar gol\n \n Brezilya tak...,9
7432,Müslüm Gürses sahada unutulmadı\nMüslüm Gürses...,3
27525,Kartal görünümlü Kanarya\nF.Bahçe kadrosunda b...,9


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 29394
})
Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 6299
})
Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 6299
})


In [ ]:
from transformers import AutoTokenizer

model_name = "dbmdz/bert-base-turkish-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer yüklendi")

Tokenizer yüklendi


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(train_dataset)

Map:   0%|          | 0/29394 [00:00<?, ? examples/s]

Map:   0%|          | 0/6299 [00:00<?, ? examples/s]

Map:   0%|          | 0/6299 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 29394
})


In [ ]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [ ]:
from transformers import AutoModelForSequenceClassification

num_labels = 13

model = AutoModelForSequenceClassification.from_pretrained(
    "dbmdz/bert-base-turkish-cased",
    num_labels=num_labels
)

print("Model yüklendi")

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model yüklendi


In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    num_train_epochs=1,

    logging_steps=500,
    fp16=True
)

print("Training ayarları hazır")

Training ayarları hazır


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Trainer hazır")

Trainer hazır


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.226280,0.876246,0.701222


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=919, training_loss=1.0827398953941105, metrics={'train_runtime': 221.4373, 'train_samples_per_second': 132.742, 'train_steps_per_second': 4.15, 'total_flos': 1933662548786688.0, 'train_loss': 1.0827398953941105, 'epoch': 1.0})

In [ ]:
trainer.evaluate(test_dataset)

{'eval_loss': 0.8897892832756042,
 'eval_accuracy': 0.6948722019368153,
 'eval_runtime': 12.6264,
 'eval_samples_per_second': 498.877,
 'eval_steps_per_second': 15.602,
 'epoch': 1.0}

In [ ]:
trainer.save_model("/content/news_model")
tokenizer.save_pretrained("/content/news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/news_model/tokenizer_config.json',
 '/content/news_model/tokenizer.json')

In [ ]:
from datasets import load_dataset

sentiment_dataset = load_dataset("winvoker/turkish-sentiment-analysis-dataset")
sentiment_dataset

README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/76.1M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/440679 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/48965 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'dataset'],
        num_rows: 440679
    })
    test: Dataset({
        features: ['text', 'label', 'dataset'],
        num_rows: 48965
    })
})

In [ ]:
sentiment_df = sentiment_dataset["train"].to_pandas()
sentiment_df.head()

,text,label,dataset
0,ürünü hepsiburadadan alalı 3 hafta oldu. orjin...,Positive,urun_yorumlari
1,"ürünlerden çok memnunum, kesinlikle herkese ta...",Positive,urun_yorumlari
2,"hızlı kargo, temiz alışveriş.teşekkür ederim.",Positive,urun_yorumlari
3,Çünkü aranan tapınak bu bölgededir .,Notr,wiki
4,bu telefonu başlıca alma nedenlerim ise elimde...,Positive,urun_yorumlari


In [ ]:
sentiment_df.columns
sentiment_df.head()

,text,label,dataset
0,ürünü hepsiburadadan alalı 3 hafta oldu. orjin...,Positive,urun_yorumlari
1,"ürünlerden çok memnunum, kesinlikle herkese ta...",Positive,urun_yorumlari
2,"hızlı kargo, temiz alışveriş.teşekkür ederim.",Positive,urun_yorumlari
3,Çünkü aranan tapınak bu bölgededir .,Notr,wiki
4,bu telefonu başlıca alma nedenlerim ise elimde...,Positive,urun_yorumlari


In [ ]:
from sklearn.preprocessing import LabelEncoder

le2 = LabelEncoder()

sentiment_df["label_id"] = le2.fit_transform(sentiment_df["label"])

print(sentiment_df["label"].unique())
print(sentiment_df["label_id"].unique())

sentiment_df.head()

['Positive' 'Notr' 'Negative']
[2 1 0]


,text,label,dataset,label_id
0,ürünü hepsiburadadan alalı 3 hafta oldu. orjin...,Positive,urun_yorumlari,2
1,"ürünlerden çok memnunum, kesinlikle herkese ta...",Positive,urun_yorumlari,2
2,"hızlı kargo, temiz alışveriş.teşekkür ederim.",Positive,urun_yorumlari,2
3,Çünkü aranan tapınak bu bölgededir .,Notr,wiki,1
4,bu telefonu başlıca alma nedenlerim ise elimde...,Positive,urun_yorumlari,2


In [ ]:
le2 = LabelEncoder()
...

In [ ]:
from sklearn.model_selection import train_test_split

train_df2, temp_df2 = train_test_split(
    sentiment_df,
    test_size=0.3,
    random_state=42,
    stratify=sentiment_df["label_id"]
)

val_df2, test_df2 = train_test_split(
    temp_df2,
    test_size=0.5,
    random_state=42,
    stratify=temp_df2["label_id"]
)

print("Train:", train_df2.shape)
print("Validation:", val_df2.shape)
print("Test:", test_df2.shape)

Train: (308475, 4)
Validation: (66102, 4)
Test: (66102, 4)


In [ ]:
sentiment_small = sentiment_df.groupby("label", group_keys=False).apply(
    lambda x: x.sample(15000, random_state=42)
).reset_index(drop=True)

print(sentiment_small.shape)
sentiment_small["label"].value_counts()

(45000, 4)


/tmp/ipykernel_1044/2757381481.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentiment_small = sentiment_df.groupby("label", group_keys=False).apply(


,count
label,
Negative,15000
Notr,15000
Positive,15000


In [ ]:
train_df2, temp_df2 = train_test_split(
    sentiment_small,
    test_size=0.3,
    random_state=42,
    stratify=sentiment_small["label_id"]
)

In [ ]:
sentiment_small = ...

In [ ]:
val_df2, test_df2 = train_test_split(
    temp_df2,
    test_size=0.5,
    random_state=42,
    stratify=temp_df2["label_id"]
)

print("Train:", train_df2.shape)
print("Validation:", val_df2.shape)
print("Test:", test_df2.shape)

Train: (31500, 4)
Validation: (6750, 4)
Test: (6750, 4)


In [ ]:
train_df2 = train_df2[["text", "label_id"]].rename(columns={"label_id": "label"})
val_df2 = val_df2[["text", "label_id"]].rename(columns={"label_id": "label"})
test_df2 = test_df2[["text", "label_id"]].rename(columns={"label_id": "label"})

train_df2.head()

,text,label
3966,rezalet..11 eylül filimleri arasında izlediği...,0
43854,"Bu filme 3 veriyorsan ben sana daha ne diyim,...",2
24926,Yoksulluk içinde büyümüştür .,1
23159,Büyük ölçüde uçuş eğitimi ve savaş zamanı üret...,1
5193,rezalet bir işletmecilik eylül tarihinde ote...,0


In [ ]:
from datasets import Dataset

train_dataset2 = Dataset.from_pandas(train_df2)
val_dataset2 = Dataset.from_pandas(val_df2)
test_dataset2 = Dataset.from_pandas(test_df2)

print(train_dataset2)
print(val_dataset2)
print(test_dataset2)

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 31500
})
Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 6750
})
Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 6750
})


In [ ]:
def tokenize_function2(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset2 = train_dataset2.map(tokenize_function2, batched=True)
val_dataset2 = val_dataset2.map(tokenize_function2, batched=True)
test_dataset2 = test_dataset2.map(tokenize_function2, batched=True)

Map:   0%|          | 0/31500 [00:00<?, ? examples/s]

Map:   0%|          | 0/6750 [00:00<?, ? examples/s]

Map:   0%|          | 0/6750 [00:00<?, ? examples/s]

In [ ]:
train_dataset2.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset2.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset2.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
from transformers import AutoModelForSequenceClassification

sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    "dbmdz/bert-base-turkish-cased",
    num_labels=3
)

print("Sentiment modeli yüklendi")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Sentiment modeli yüklendi


In [ ]:
import evaluate
import numpy as np

accuracy2 = evaluate.load("accuracy")

def compute_metrics2(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy2.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import TrainingArguments

sentiment_training_args = TrainingArguments(
    output_dir="./sentiment_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    logging_steps=500,
    fp16=True
)

print("Sentiment training ayarları hazır")

Sentiment training ayarları hazır


In [ ]:
from transformers import Trainer

sentiment_trainer = Trainer(
    model=sentiment_model,
    args=sentiment_training_args,
    train_dataset=train_dataset2,
    eval_dataset=val_dataset2,
    compute_metrics=compute_metrics2
)

print("Sentiment Trainer hazır")

Sentiment Trainer hazır


In [ ]:
sentiment_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.244968,0.153276,0.945926


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=985, training_loss=0.20007422587593196, metrics={'train_runtime': 224.9834, 'train_samples_per_second': 140.01, 'train_steps_per_second': 4.378, 'total_flos': 2072018164608000.0, 'train_loss': 0.20007422587593196, 'epoch': 1.0})

In [ ]:
sentiment_trainer.evaluate(test_dataset2)

{'eval_loss': 0.14441899955272675,
 'eval_accuracy': 0.9488888888888889,
 'eval_runtime': 13.0274,
 'eval_samples_per_second': 518.139,
 'eval_steps_per_second': 16.197,
 'epoch': 1.0}

In [ ]:
sentiment_trainer.save_model("/content/sentiment_model")
tokenizer.save_pretrained("/content/sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/sentiment_model/tokenizer_config.json',
 '/content/sentiment_model/tokenizer.json')

In [ ]:
import gradio as gr
import torch
from transformers import pipeline

In [ ]:
news_classifier = pipeline(
    "text-classification",
    model="/content/news_model",
    tokenizer="/content/news_model"
)

sentiment_classifier = pipeline(
    "text-classification",
    model="/content/sentiment_model",
    tokenizer="/content/sentiment_model"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
def predict_news(text):
    result = news_classifier(text)[0]
    return result["label"]

def predict_sentiment(text):
    result = sentiment_classifier(text)[0]
    return result["label"]

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# Türkçe NLP Projesi")

    with gr.Tab("Haber Sınıflandırma"):
        inp1 = gr.Textbox(label="Metin Gir")
        out1 = gr.Textbox(label="Kategori")
        btn1 = gr.Button("Tahmin Et")
        btn1.click(predict_news, inp1, out1)

    with gr.Tab("Duygu Analizi"):
        inp2 = gr.Textbox(label="Yorum Gir")
        out2 = gr.Textbox(label="Sonuç")
        btn2 = gr.Button("Tahmin Et")
        btn2.click(predict_sentiment, inp2, out2)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d8c3bb8af2cfc6f8d5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
news_labels = {
    "LABEL_0":"dunya",
    "LABEL_1":"ekonomi",
    "LABEL_2":"genel",
    "LABEL_3":"guncel",
    "LABEL_4":"kultur-sanat",
    "LABEL_5":"magazin",
    "LABEL_6":"planet",
    "LABEL_7":"saglik",
    "LABEL_8":"siyaset",
    "LABEL_9":"spor",
    "LABEL_10":"teknoloji",
    "LABEL_11":"turkiye",
    "LABEL_12":"yasam"
}

sentiment_labels = {
    "LABEL_0":"Negative 😡",
    "LABEL_1":"Neutral 😐",
    "LABEL_2":"Positive 😍"
}

def predict_news(text):
    result = news_classifier(text)[0]
    return news_labels[result["label"]]

def predict_sentiment(text):
    result = sentiment_classifier(text)[0]
    return sentiment_labels[result["label"]]

In [ ]:
demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d8c3bb8af2cfc6f8d5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
sentiment_labels = {
    "LABEL_0": "Negative 😡",
    "LABEL_1": "Neutral 😐",
    "LABEL_2": "Positive 😍"
}

In [ ]:
print(label_encoder.classes_)

['dunya' 'ekonomi' 'genel' 'guncel' 'kultur-sanat' 'magazin' 'planet'
 'saglik' 'siyaset' 'spor' 'teknoloji' 'turkiye' 'yasam']


In [ ]:
news_labels = {
    "LABEL_0":"dunya 🌍",
    "LABEL_1":"ekonomi 💰",
    "LABEL_2":"genel 📰",
    "LABEL_3":"guncel ⏰",
    "LABEL_4":"kultur-sanat 🎭",
    "LABEL_5":"magazin ✨",
    "LABEL_6":"planet 🌎",
    "LABEL_7":"saglik 🏥",
    "LABEL_8":"siyaset 🏛️",
    "LABEL_9":"spor ⚽",
    "LABEL_10":"teknoloji 💻",
    "LABEL_11":"turkiye 🇹🇷",
    "LABEL_12":"yasam 🌿"
}

In [ ]:
sentiment_labels = {
    "LABEL_0":"Negative 😡",
    "LABEL_1":"Neutral 😐",
    "LABEL_2":"Positive 😍"
}

In [ ]:
def predict_news(text):
    result = news_classifier(text)[0]
    return news_labels[result["label"]]

def predict_sentiment(text):
    result = sentiment_classifier(text)[0]
    return sentiment_labels[result["label"]]

In [ ]:
demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d8c3bb8af2cfc6f8d5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
news_labels = {
    "LABEL_0":"dunya 🌍",
    "LABEL_1":"ekonomi 💰",
    "LABEL_2":"genel 📰",
    "LABEL_3":"guncel ⏰",
    "LABEL_4":"kultur-sanat 🎭",
    "LABEL_5":"magazin ✨",
    "LABEL_6":"planet 🌎",
    "LABEL_7":"saglik 🏥",
    "LABEL_8":"siyaset 🏛️",
    "LABEL_9":"spor ⚽",
    "LABEL_10":"teknoloji 💻",
    "LABEL_11":"turkiye 🇹🇷",
    "LABEL_12":"yasam 🌿"
}

sentiment_labels = {
    "LABEL_0":"Negative 😡",
    "LABEL_1":"Neutral 😐",
    "LABEL_2":"Positive 😍"
}

def predict_news(text):
    result = news_classifier(text)[0]
    raw_label = result["label"]
    return news_labels.get(raw_label, raw_label)

def predict_sentiment(text):
    result = sentiment_classifier(text)[0]
    raw_label = result["label"]
    return sentiment_labels.get(raw_label, raw_label)

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# Türkçe NLP Projesi")

    with gr.Tab("Haber Sınıflandırma"):
        inp1 = gr.Textbox(label="Metin Gir")
        out1 = gr.Textbox(label="Kategori")
        btn1 = gr.Button("Tahmin Et")
        btn1.click(fn=predict_news, inputs=inp1, outputs=out1)

    with gr.Tab("Duygu Analizi"):
        inp2 = gr.Textbox(label="Yorum Gir")
        out2 = gr.Textbox(label="Sonuç")
        btn2 = gr.Button("Tahmin Et")
        btn2.click(fn=predict_sentiment, inputs=inp2, outputs=out2)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://89f0cae3d1b4233e48.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

project_path = "/content/drive/MyDrive/CIF425_Turkce_NLP_Projesi"

os.makedirs(project_path, exist_ok=True)
os.makedirs(project_path + "/news_model", exist_ok=True)
os.makedirs(project_path + "/sentiment_model", exist_ok=True)

print("Proje klasörü oluşturuldu.")

Proje klasörü oluşturuldu.


In [ ]:
!cp -r /content/news_model/* /content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/news_model/
!cp -r /content/sentiment_model/* /content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/sentiment_model/

print("Modeller Drive'a kaydedildi.")

Modeller Drive'a kaydedildi.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import pipeline

news_classifier = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/news_model",
    tokenizer="/content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/news_model"
)

sentiment_classifier = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/sentiment_model",
    tokenizer="/content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/sentiment_model"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bd18fc112103be6c7f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def predict_news(text):
    result = news_classifier(text)
    return result[0]["label"]

def predict_sentiment(text):
    result = sentiment_classifier(text)
    return result[0]["label"]

with gr.Blocks() as demo:
    gr.Markdown("## Türkçe NLP Projesi")

    with gr.Tab("Haber Sınıflandırma"):
        news_input = gr.Textbox(label="Metin Gir")
        news_output = gr.Textbox(label="Kategori")
        news_btn = gr.Button("Tahmin Et")
        news_btn.click(predict_news, inputs=news_input, outputs=news_output)

    with gr.Tab("Duygu Analizi"):
        sent_input = gr.Textbox(label="Yorum Gir")
        sent_output = gr.Textbox(label="Sonuç")
        sent_btn = gr.Button("Tahmin Et")
        sent_btn.click(predict_sentiment, inputs=sent_input, outputs=sent_output)

In [ ]:
from transformers import AutoConfig

news_model_path = "/content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/news_model"
sentiment_model_path = "/content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/sentiment_model"

news_id2label = {
    0: "dunya 🌍",
    1: "ekonomi 💰",
    2: "genel 📰",
    3: "guncel ⏰",
    4: "kultur-sanat 🎭",
    5: "magazin ✨",
    6: "planet 🌎",
    7: "saglik 🏥",
    8: "siyaset 🏛️",
    9: "spor ⚽",
    10: "teknoloji 💻",
    11: "turkiye 🇹🇷",
    12: "yasam 🌿"
}

sentiment_id2label = {
    0: "Negative 😡",
    1: "Neutral 😐",
    2: "Positive 😍"
}

news_config = AutoConfig.from_pretrained(news_model_path)
news_config.id2label = news_id2label
news_config.label2id = {v: k for k, v in news_id2label.items()}
news_config.save_pretrained(news_model_path)

sentiment_config = AutoConfig.from_pretrained(sentiment_model_path)
sentiment_config.id2label = sentiment_id2label
sentiment_config.label2id = {v: k for k, v in sentiment_id2label.items()}
sentiment_config.save_pretrained(sentiment_model_path)

print("Drive'daki model label isimleri güncellendi.")

Drive'daki model label isimleri güncellendi.


In [ ]:
from transformers import pipeline

news_classifier = pipeline(
    "text-classification",
    model=news_model_path,
    tokenizer=news_model_path
)

sentiment_classifier = pipeline(
    "text-classification",
    model=sentiment_model_path,
    tokenizer=sentiment_model_path
)

print("Modeller güncel label isimleriyle yüklendi.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modeller güncel label isimleriyle yüklendi.


In [ ]:
def predict_news(text):
    result = news_classifier(text)
    return result[0]["label"]

def predict_sentiment(text):
    result = sentiment_classifier(text)
    return result[0]["label"]

In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## Türkçe NLP Projesi")

    with gr.Tab("Haber Sınıflandırma"):
        news_input = gr.Textbox(label="Metin Gir")
        news_output = gr.Textbox(label="Kategori")
        news_btn = gr.Button("Tahmin Et")
        news_btn.click(predict_news, inputs=news_input, outputs=news_output)

    with gr.Tab("Duygu Analizi"):
        sent_input = gr.Textbox(label="Yorum Gir")
        sent_output = gr.Textbox(label="Sonuç")
        sent_btn = gr.Button("Tahmin Et")
        sent_btn.click(predict_sentiment, inputs=sent_input, outputs=sent_output)

In [ ]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d68ff66959bedc226d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers gradio

In [ ]:
from transformers import pipeline

news_model_path = "/content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/news_model"
sentiment_model_path = "/content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/sentiment_model"

news_classifier = pipeline(
    "text-classification",
    model=news_model_path,
    tokenizer=news_model_path
)

sentiment_classifier = pipeline(
    "text-classification",
    model=sentiment_model_path,
    tokenizer=sentiment_model_path
)

print("Modeller yüklendi.")

Loading weights:   0%|          | 0/201 [00:02<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Modeller yüklendi.


In [ ]:
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## Türkçe NLP Projesi")

    with gr.Tab("Haber Sınıflandırma"):
        news_input = gr.Textbox(label="Metin Gir")
        news_output = gr.Textbox(label="Kategori")
        news_btn = gr.Button("Tahmin Et")
        news_btn.click(predict_news, inputs=news_input, outputs=news_output)

    with gr.Tab("Duygu Analizi"):
        sent_input = gr.Textbox(label="Yorum Gir")
        sent_output = gr.Textbox(label="Sonuç")
        sent_btn = gr.Button("Tahmin Et")
        sent_btn.click(predict_sentiment, inputs=sent_input, outputs=sent_output)

In [ ]:
def predict_news(text):
    result = news_classifier(text)
    return result[0]["label"]

def predict_sentiment(text):
    result = sentiment_classifier(text)
    return result[0]["label"]

In [ ]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f82ce079ed2aa63a0f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def predict_sentiment(text):
    result = sentiment_classifier(text)

    label = result[0]["label"]
    score = result[0]["score"]

    confidence = round(score * 100, 2)

    return f"{label} (%{confidence})"

In [ ]:
def predict_news(text):
    result = news_classifier(text)

    label = result[0]["label"]
    score = result[0]["score"]

    confidence = round(score * 100, 2)

    return f"{label} (%{confidence})"

In [ ]:
demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f82ce079ed2aa63a0f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
def predict_news(text):
    result = news_classifier(text)[0]
    label = result["label"]
    score = round(result["score"] * 100, 2)
    return f"{label} (%{score})"

def predict_sentiment(text):
    result = sentiment_classifier(text)[0]
    label = result["label"]
    score = round(result["score"] * 100, 2)
    return f"{label} (%{score})"

import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## Türkçe NLP Projesi")

    with gr.Tab("Haber Sınıflandırma"):
        news_input = gr.Textbox(label="Metin Gir")
        news_output = gr.Textbox(label="Kategori")
        news_btn = gr.Button("Tahmin Et")
        news_btn.click(predict_news, inputs=news_input, outputs=news_output)

    with gr.Tab("Duygu Analizi"):
        sent_input = gr.Textbox(label="Yorum Gir")
        sent_output = gr.Textbox(label="Sonuç")
        sent_btn = gr.Button("Tahmin Et")
        sent_btn.click(predict_sentiment, inputs=sent_input, outputs=sent_output)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1c121b5d29303ca035.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def predict_both(text):
    news_result = news_classifier(text)[0]
    sentiment_result = sentiment_classifier(text)[0]

    news_label = news_result["label"]
    news_score = round(news_result["score"] * 100, 2)

    sentiment_label = sentiment_result["label"]
    sentiment_score = round(sentiment_result["score"] * 100, 2)

    return (
        f"{news_label} (%{news_score})",
        f"{sentiment_label} (%{sentiment_score})"
    )

with gr.Blocks() as demo:
    gr.Markdown("# Türkçe Haber Analiz Uygulaması")
    gr.Markdown("Metni girin; model hem haber kategorisini hem de duygu durumunu tahmin etsin.")

    text_input = gr.Textbox(
        label="Haber / Metin Gir",
        lines=5
    )

    predict_btn = gr.Button("Analiz Et")

    category_output = gr.Textbox(label="Tahmin Edilen Haber Kategorisi")
    sentiment_output = gr.Textbox(label="Tahmin Edilen Duygu")

    predict_btn.click(
        fn=predict_both,
        inputs=text_input,
        outputs=[category_output, sentiment_output]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://916c376f155af5660f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

examples = [
    ["Galatasaray derbiyi kazandı ve taraftarlar büyük sevinç yaşadı"],
    ["Merkez Bankası faiz oranlarını artırma kararı aldı"],
    ["Apple yeni yapay zeka destekli telefonunu tanıttı"],
    ["Bu ürün harika çok memnun kaldım"],
    ["Hiç memnun kalmadım ürün çok kötüydü"],
    ["Toplantı yarın saat 10:00'da yapılacak"]
]

def predict_both(text):
    news_result = news_classifier(text)[0]
    sentiment_result = sentiment_classifier(text)[0]

    news_label = news_result["label"]
    news_score = round(news_result["score"] * 100, 2)

    sentiment_label = sentiment_result["label"]
    sentiment_score = round(sentiment_result["score"] * 100, 2)

    news_output = f"📰 Kategori: {news_label}\n📊 Güven Skoru: %{news_score}"
    sentiment_output = f"😊 Duygu: {sentiment_label}\n📊 Güven Skoru: %{sentiment_score}"

    summary = (
        f"Bu metin model tarafından **{news_label}** kategorisine yakın bulundu. "
        f"Duygu analizi sonucu ise **{sentiment_label}** olarak tahmin edildi."
    )

    return news_output, sentiment_output, summary


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🇹🇷 Türkçe Haber Analiz Uygulaması
        BERTurk ile geliştirilmiş bu uygulama, girilen metin için hem **haber kategorisi** hem de **duygu analizi** tahmini yapar.
        """
    )

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="Metin Gir",
                placeholder="Örnek: Galatasaray derbiyi kazandı ve taraftarlar büyük sevinç yaşadı.",
                lines=6
            )

            analyze_btn = gr.Button("🚀 Analiz Et", variant="primary")

        with gr.Column(scale=1):
            category_output = gr.Textbox(label="Haber Kategorisi", lines=2)
            sentiment_output = gr.Textbox(label="Duygu Analizi", lines=2)

    summary_output = gr.Markdown(label="Model Yorumu")

    gr.Examples(
        examples=examples,
        inputs=text_input
    )

    analyze_btn.click(
        fn=predict_both,
        inputs=text_input,
        outputs=[category_output, sentiment_output, summary_output]
    )

demo.launch(share=True)

/tmp/ipykernel_5447/1817895752.py:33: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f5cdd5fc5a2c176255.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

# Örnek veriler
examples = [
    ["Galatasaray derbiyi kazandı ve taraftarlar büyük sevinç yaşadı"],
    ["Merkez Bankası faiz oranlarını artırma kararı aldı"],
    ["Apple yeni yapay zeka destekli telefonunu tanıttı"],
    ["Bu ürün harika çok memnun kaldım"],
    ["Hiç memnun kalmadım ürün çok kötüydü"],
    ["Toplantı yarın saat 10:00'da yapılacak"]
]

def predict_both(text):
    # Not: news_classifier ve sentiment_classifier fonksiyonlarınızın tanımlı olması gerekir.
    # Burada örnek olması için statik veriler döndürüyorum:
    news_label, news_score = "Ekonomi", 92.4
    sentiment_label, sentiment_score = "Pozitif", 88.1

    news_html = f"""
    <div style='background-color: #f0f7ff; padding: 15px; border-left: 5px solid #2196F3; border-radius: 5px; margin-bottom: 10px;'>
        <h4 style='margin: 0; color: #1565C0;'>📰 Haber Kategorisi</h4>
        <p style='font-size: 20px; font-weight: bold; margin: 5px 0;'>{news_label}</p>
        <span style='color: #555;'>Güven Skoru: %{news_score}</span>
    </div>
    """

    sentiment_html = f"""
    <div style='background-color: #f6fff0; padding: 15px; border-left: 5px solid #4CAF50; border-radius: 5px;'>
        <h4 style='margin: 0; color: #2E7D32;'>😊 Duygu Analizi</h4>
        <p style='font-size: 20px; font-weight: bold; margin: 5px 0;'>{sentiment_label}</p>
        <span style='color: #555;'>Güven Skoru: %{sentiment_score}</span>
    </div>
    """

    summary = f"### 🤖 Analiz Özeti\nMetin, **{news_label}** kategorisinde değerlendirildi ve **{sentiment_label}** bir tonda olduğu tespit edildi."

    return news_html, sentiment_html, summary

# Arayüz Tasarımı
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate")) as demo:

    gr.HTML(
        """
        <div style="text-align: center; max-width: 800px; margin: 0 auto; padding: 20px;">
            <h1 style="font-weight: 900; font-size: 2.5rem; margin-bottom: 0.5rem;">
                🇹🇷 BERTurk <span style="color: #2196F3;">Analiz Merkezi</span>
            </h1>
            <p style="font-size: 1.1rem; color: #666;">Modern Türkçe Doğal Dil İşleme Paneli</p>
        </div>
        """
    )

    with gr.Row():
        # Sol Kolon
        with gr.Column(scale=3):
            text_input = gr.Textbox(
                label="Giriş Metni",
                placeholder="Analiz edilecek içeriği buraya yazın...",
                lines=8
            )
            analyze_btn = gr.Button("🚀 Analiz Et", variant="primary")

            gr.Examples(examples=examples, inputs=text_input)

        # Sağ Kolon (Hata Alınan Kısım Düzeltildi)
        with gr.Column(scale=2):
            with gr.Group():
                gr.Markdown("### 📊 Analiz Sonuçları")
                category_output = gr.HTML()
                sentiment_output = gr.HTML()

            with gr.Accordion("Detaylı Model Yorumu", open=True):
                summary_output = gr.Markdown()

    analyze_btn.click(
        fn=predict_both,
        inputs=text_input,
        outputs=[category_output, sentiment_output, summary_output]
    )

if __name__ == "__main__":
    demo.launch()

/tmp/ipykernel_5447/2869612562.py:40: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="slate")) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eec89f310735b43d86.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

def predict_both(text):
    # Model çıktılarını simüle ediyoruz (Burayı kendi modellerine bağla)
    news_label, news_score = "Teknoloji", 94.5
    sentiment_label, sentiment_score = "Pozitif", 82.0

    # Grafiksel ilerleme çubuğu (Progress Bar) fonksiyonu
    def create_bar(score, color):
        return f"""
        <div style="width: 100%; background-color: #e0e0e0; border-radius: 10px; height: 10px; margin-top: 5px;">
            <div style="width: {score}%; background-color: {color}; height: 10px; border-radius: 10px;"></div>
        </div>
        """

    # Kategori Kartı (Mavi Tonları)
    news_html = f"""
    <div style="background: white; border: 1px solid #e0e0e0; padding: 15px; border-radius: 12px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
        <div style="display: flex; justify-content: space-between; align-items: center;">
            <span style="color: #1a73e8; font-weight: bold; font-size: 14px;">📰 KATEGORİ</span>
            <span style="font-weight: bold; color: #1a73e8;">%{news_score}</span>
        </div>
        <div style="font-size: 22px; font-weight: 800; margin: 8px 0; color: #202124;">{news_label}</div>
        {create_bar(news_score, '#1a73e8')}
    </div>
    """

    # Duygu Kartı (Yeşil/Doğal Tonlar)
    sentiment_html = f"""
    <div style="background: white; border: 1px solid #e0e0e0; padding: 15px; border-radius: 12px; box-shadow: 0 2px 4px rgba(0,0,0,0.05); margin-top: 10px;">
        <div style="display: flex; justify-content: space-between; align-items: center;">
            <span style="color: #34a853; font-weight: bold; font-size: 14px;">😊 DUYGU ANALİZİ</span>
            <span style="font-weight: bold; color: #34a853;">%{sentiment_score}</span>
        </div>
        <div style="font-size: 22px; font-weight: 800; margin: 8px 0; color: #202124;">{sentiment_label}</div>
        {create_bar(sentiment_score, '#34a853')}
    </div>
    """

    summary = f"**Özet:** Analiz sonucuna göre metin %{news_score} güvenle **{news_label}** içeriği taşımakta ve **{sentiment_label}** bir tutum sergilemektedir."

    return news_html, sentiment_html, summary

# Tema ayarları
theme = gr.themes.Default(
    primary_hue="blue",
    secondary_hue="gray",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui"],
)

with gr.Blocks(theme=theme, css=".gradio-container {background-color: #f8f9fa}") as demo:

    with gr.Column(elem_id="main-container"):
        gr.HTML(
            """
            <div style="text-align: center; padding: 30px 0;">
                <h1 style="color: #1a73e8; font-size: 2.5em; font-weight: 900; margin-bottom: 10px;">BERTurk <span style="color: #202124;">Intelligence</span></h1>
                <p style="color: #5f6368; font-size: 1.1em;">Türkçe Metinler İçin Gelişmiş Klasifikasyon ve Duygu Skorlama</p>
            </div>
            """
        )

        with gr.Row():
            # Giriş Alanı
            with gr.Column(scale=3):
                text_input = gr.Textbox(
                    label="Metin Girişi",
                    placeholder="Analiz edilecek içeriği buraya bırakın...",
                    lines=10,
                    container=True
                )
                with gr.Row():
                    clear_btn = gr.Button("Temizle", variant="secondary")
                    submit_btn = gr.Button("Analizi Başlat", variant="primary")

                gr.Examples(
                    examples=[["Borsada bugün rekor yükseliş kaydedildi."], ["Yeni çıkan filmi hiç beğenmedim, vakit kaybıydı."]],
                    inputs=text_input
                )

            # Sonuç Alanı
            with gr.Column(scale=2):
                with gr.Group(): # image_cb76b9.png'deki hatayı önlemek için Group kullanıldı
                    gr.Markdown("### 📊 Canlı Analiz Skorları")
                    news_out = gr.HTML()
                    sent_out = gr.HTML()

                with gr.Box(style="margin-top: 15px;"):
                    summary_out = gr.Markdown("Sonuçlar burada görünecek...")

    # Buton Fonksiyonları
    submit_btn.click(
        fn=predict_both,
        inputs=text_input,
        outputs=[news_out, sent_out, summary_out]
    )
    clear_btn.click(lambda: [None, None, ""], outputs=[news_out, sent_out, summary_out])

demo.launch()

/tmp/ipykernel_5447/3188262315.py:51: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=".gradio-container {background-color: #f8f9fa}") as demo:
/tmp/ipykernel_5447/3188262315.py:51: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=".gradio-container {background-color: #f8f9fa}") as demo:


AttributeError: module 'gradio' has no attribute 'Box'

In [ ]:
import gradio as gr

def predict_both(text):
    # Simüle edilmiş veriler
    news_label, news_score = "Siyaset", 89.2
    sentiment_label, sentiment_score = "Nötr", 75.5

    def create_bar(score, color):
        return f"""
        <div style="width: 100%; background-color: #e0e0e0; border-radius: 10px; height: 8px; margin-top: 5px;">
            <div style="width: {score}%; background-color: {color}; height: 8px; border-radius: 10px;"></div>
        </div>
        """

    news_html = f"""
    <div style="background: white; border: 1px solid #eef2f6; padding: 20px; border-radius: 15px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
        <div style="display: flex; justify-content: space-between; align-items: center;">
            <span style="color: #4f46e5; font-weight: 600; font-size: 12px; letter-spacing: 0.05em;">HABER KATEGORİSİ</span>
            <span style="font-weight: bold; color: #4f46e5; background: #eef2f6; padding: 2px 8px; border-radius: 10px; font-size: 12px;">%{news_score}</span>
        </div>
        <div style="font-size: 24px; font-weight: 800; margin: 10px 0; color: #1e293b;">{news_label}</div>
        {create_bar(news_score, '#4f46e5')}
    </div>
    """

    sentiment_html = f"""
    <div style="background: white; border: 1px solid #f0fdf4; padding: 20px; border-radius: 15px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); margin-top: 15px;">
        <div style="display: flex; justify-content: space-between; align-items: center;">
            <span style="color: #16a34a; font-weight: 600; font-size: 12px; letter-spacing: 0.05em;">DUYGU DURUMU</span>
            <span style="font-weight: bold; color: #16a34a; background: #f0fdf4; padding: 2px 8px; border-radius: 10px; font-size: 12px;">%{sentiment_score}</span>
        </div>
        <div style="font-size: 24px; font-weight: 800; margin: 10px 0; color: #1e293b;">{sentiment_label}</div>
        {create_bar(sentiment_score, '#16a34a')}
    </div>
    """

    summary = f"### 💡 Analiz Özeti\nModelimiz bu metni **{news_label}** kategorisinde sınıflandırdı. Metnin genel duygu tonu ise **{sentiment_label}** olarak belirlendi."

    return news_html, sentiment_html, summary

# Profesyonel Tema Tasarımı
custom_theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="slate",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Plus Jakarta Sans"), "ui-sans-serif"]
)

# Gelecek sürümlerle uyumlu (Deprecation hatasız) yapı
with gr.Blocks(theme=custom_theme) as demo:

    with gr.Container():
        gr.HTML(
            """
            <div style="text-align: center; padding: 40px 0;">
                <h1 style="color: #4f46e5; font-size: 2.8em; font-weight: 900; margin-bottom: 8px; letter-spacing: -0.02em;">
                    BERTurk <span style="color: #1e293b;">NLP Engine</span>
                </h1>
                <p style="color: #64748b; font-size: 1.2em; max-width: 600px; margin: 0 auto;">
                    Kurumsal düzeyde Türkçe metin analizi, kategori tahmini ve duygu skorlama paneli.
                </p>
            </div>
            """
        )

        with gr.Row(variant="panel"):
            # Giriş Alanı
            with gr.Column(scale=3):
                text_input = gr.Textbox(
                    label="Analiz Edilecek Metin",
                    placeholder="İçeriği buraya girin...",
                    lines=10,
                    max_lines=15
                )
                with gr.Row():
                    clear_btn = gr.Button("🗑️ Formu Temizle", variant="secondary")
                    submit_btn = gr.Button("🚀 Analizi Başlat", variant="primary")

                gr.Examples(
                    examples=[["Hükümetten yeni vergi düzenlemesi açıklaması geldi."], ["Bu hizmetten hiç memnun kalmadım, çok yavaştı."]],
                    inputs=text_input,
                    label="Hızlı Test Örnekleri"
                )

            # Sonuç Alanı
            with gr.Column(scale=2):
                with gr.Group():
                    gr.Markdown("### 📈 Canlı Skor Paneli")
                    news_out = gr.HTML()
                    sent_out = gr.HTML()

                # gr.Box yerine modern Column variant kullanımı
                with gr.Column(variant="compact", elem_id="summary_box"):
                    summary_out = gr.Markdown("Sonuç bekleniyor...")

    # Event Dinleyicileri
    submit_btn.click(
        fn=predict_both,
        inputs=text_input,
        outputs=[news_out, sent_out, summary_out],
        show_progress="minimal"
    )

    clear_btn.click(
        fn=lambda: (None, None, "Sonuç bekleniyor..."),
        outputs=[news_out, sent_out, summary_out]
    )

# CSS ve Temayı launch kısmına taşıyarak uyarıları gideriyoruz
demo.launch(
    show_api=False,
    debug=False
)

/tmp/ipykernel_5447/3743647535.py:50: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme) as demo:


AttributeError: module 'gradio' has no attribute 'Container'

In [ ]:
import gradio as gr

def predict_both(text):
    # Model simülasyonu
    news_label, news_score = "Siyaset", 89.2
    sentiment_label, sentiment_score = "Nötr", 75.5

    # Dinamik Progress Bar (İlerleme Çubuğu)
    def create_bar(score, color):
        return f"""
        <div style="width: 100%; background-color: #e5e7eb; border-radius: 10px; height: 8px; margin-top: 5px;">
            <div style="width: {score}%; background-color: {color}; height: 8px; border-radius: 10px; transition: width 0.5s ease-in-out;"></div>
        </div>
        """

    # Kategori Kartı
    news_html = f"""
    <div style="background: white; border: 1px solid #e2e8f0; padding: 16px; border-radius: 12px; margin-bottom: 12px;">
        <div style="display: flex; justify-content: space-between; align-items: center;">
            <span style="color: #6366f1; font-weight: 700; font-size: 11px; letter-spacing: 0.1em;">HABER KATEGORİSİ</span>
            <span style="font-weight: bold; color: #6366f1; background: #eef2ff; padding: 2px 8px; border-radius: 6px; font-size: 11px;">%{news_score}</span>
        </div>
        <div style="font-size: 22px; font-weight: 800; margin: 8px 0; color: #1e293b;">{news_label}</div>
        {create_bar(news_score, '#6366f1')}
    </div>
    """

    # Duygu Kartı
    sentiment_html = f"""
    <div style="background: white; border: 1px solid #f0fdf4; padding: 16px; border-radius: 12px;">
        <div style="display: flex; justify-content: space-between; align-items: center;">
            <span style="color: #22c55e; font-weight: 700; font-size: 11px; letter-spacing: 0.1em;">DUYGU DURUMU</span>
            <span style="font-weight: bold; color: #22c55e; background: #f0fdf4; padding: 2px 8px; border-radius: 6px; font-size: 11px;">%{sentiment_score}</span>
        </div>
        <div style="font-size: 22px; font-weight: 800; margin: 8px 0; color: #1e293b;">{sentiment_label}</div>
        {create_bar(sentiment_score, '#22c55e')}
    </div>
    """

    summary = f"### 💡 Analiz Özeti\nModelimiz bu metni **{news_label}** kategorisinde sınıflandırdı. Metnin genel duygu tonu ise **{sentiment_label}** olarak belirlendi."

    return news_html, sentiment_html, summary

# Tema yapılandırması
custom_theme = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="slate",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Plus Jakarta Sans"), "ui-sans-serif"]
)

with gr.Blocks(theme=custom_theme, title="BERTurk NLP") as demo:

    with gr.Column():
        # Header Section
        gr.HTML(
            """
            <div style="text-align: center; padding: 30px 0;">
                <h1 style="color: #4f46e5; font-size: 2.5em; font-weight: 900; margin-bottom: 5px;">
                    BERTurk <span style="color: #1e293b;">NLP Engine</span>
                </h1>
                <p style="color: #64748b; font-size: 1.1em;">Kurumsal Düzeyde Türkçe Metin Analiz Paneli</p>
            </div>
            """
        )

        with gr.Row():
            # Sol Taraf: Giriş
            with gr.Column(scale=3):
                text_input = gr.Textbox(
                    label="Analiz Edilecek Metin",
                    placeholder="Metni buraya girin...",
                    lines=9
                )
                with gr.Row():
                    clear_btn = gr.Button("🗑️ Temizle", variant="secondary")
                    submit_btn = gr.Button("🚀 Analiz Et", variant="primary")

                gr.Examples(
                    examples=[["Ekonomi dünyasında hareketli dakikalar yaşanıyor."], ["Bu yeni özelliği hiç beğenmedim."]],
                    inputs=text_input,
                    label="Hızlı Örnekler"
                )

            # Sağ Taraf: Skorlar
            with gr.Column(scale=2):
                with gr.Group():
                    gr.Markdown("### 📈 Analiz Skorları")
                    news_out = gr.HTML()
                    sent_out = gr.HTML()

                with gr.Column(variant="panel"):
                    summary_out = gr.Markdown("*Sonuçlar burada görünecek...*")

    # Fonksiyon Bağlantıları
    submit_btn.click(
        fn=predict_both,
        inputs=text_input,
        outputs=[news_out, sent_out, summary_out]
    )

    clear_btn.click(
        fn=lambda: (None, None, "*Sonuçlar burada görünecek...*"),
        outputs=[news_out, sent_out, summary_out]
    )

# Gelecek uyumluluğu için launch ayarları
if __name__ == "__main__":
    demo.launch(show_api=False)

/tmp/ipykernel_5447/1896872133.py:52: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, title="BERTurk NLP") as demo:
/tmp/ipykernel_5447/1896872133.py:109: DeprecationWarning: The 'show_api' parameter in launch() will be removed in Gradio 6.0. You will need to use the 'footer_links' parameter instead. To replicate show_api=False, In Gradio 6.0, use footer_links=['gradio', 'settings'].
  demo.launch(show_api=False)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://41eafd7c118a338366.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import pandas as pd

def predict_both_pretty(text):
    news_result = news_classifier(text, return_all_scores=True)[0]
    sentiment_result = sentiment_classifier(text, return_all_scores=True)[0]

    news_df = pd.DataFrame(news_result)
    sentiment_df = pd.DataFrame(sentiment_result)

    news_df["score"] = (news_df["score"] * 100).round(2)
    sentiment_df["score"] = (sentiment_df["score"] * 100).round(2)

    news_df = news_df.rename(columns={"label": "Kategori", "score": "Güven (%)"})
    sentiment_df = sentiment_df.rename(columns={"label": "Duygu", "score": "Güven (%)"})

    best_news = news_df.sort_values("Güven (%)", ascending=False).iloc[0]
    best_sentiment = sentiment_df.sort_values("Güven (%)", ascending=False).iloc[0]

    summary = f"""
### ✅ Analiz Sonucu

📰 **Tahmin Edilen Kategori:** `{best_news["Kategori"]}`
📊 **Kategori Güveni:** `%{best_news["Güven (%)"]}`

😊 **Tahmin Edilen Duygu:** `{best_sentiment["Duygu"]}`
📊 **Duygu Güveni:** `%{best_sentiment["Güven (%)"]}`
"""

    return summary, news_df, sentiment_df


examples = [
    ["Galatasaray derbiyi kazandı ve taraftarlar büyük sevinç yaşadı."],
    ["Merkez Bankası faiz oranlarını artırma kararı aldı."],
    ["Apple yeni yapay zeka destekli telefonunu tanıttı."],
    ["Bu ürün harika çok memnun kaldım."],
    ["Hiç memnun kalmadım ürün çok kötüydü."],
    ["Toplantı yarın saat 10:00'da yapılacak."]
]


with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="cyan")) as demo:
    gr.Markdown(
        """
        # 🇹🇷 Türkçe Haber ve Duygu Analiz Uygulaması

        Bu uygulama **BERTurk** modeli ile girilen metni analiz eder.
        Hem **haber kategorisini** hem de **duygu durumunu** tahmin eder.
        """
    )

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="📝 Metin Gir",
                placeholder="Örnek: Galatasaray derbiyi kazandı ve taraftarlar büyük sevinç yaşadı.",
                lines=7
            )

            analyze_btn = gr.Button("🚀 Analiz Et", variant="primary")

        with gr.Column(scale=1):
            summary_output = gr.Markdown(label="Sonuç")

    gr.Markdown("## 📊 Güven Skoru Grafikleri")

    with gr.Row():
        news_plot = gr.BarPlot(
            x="Kategori",
            y="Güven (%)",
            title="Haber Kategorisi Güven Skorları",
            vertical=False
        )

        sentiment_plot = gr.BarPlot(
            x="Duygu",
            y="Güven (%)",
            title="Duygu Analizi Güven Skorları",
            vertical=False
        )

    gr.Markdown("## 🧪 Hazır Örnekler")

    gr.Examples(
        examples=examples,
        inputs=text_input
    )

    analyze_btn.click(
        fn=predict_both_pretty,
        inputs=text_input,
        outputs=[summary_output, news_plot, sentiment_plot]
    )

demo.launch(share=True)

/tmp/ipykernel_5447/3500233780.py:43: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="cyan")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e2b3ce50a4066c1522.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import pandas as pd

def predict_both_pretty(text):
    news_result = news_classifier(text, return_all_scores=True)[0]
    sentiment_result = sentiment_classifier(text, return_all_scores=True)[0]

    news_df = pd.DataFrame(news_result)
    sentiment_df = pd.DataFrame(sentiment_result)

    # yüzdeye çevir
    news_df["score"] = news_df["score"] * 100
    sentiment_df["score"] = sentiment_df["score"] * 100

    # kolon isimlerini eşitle (ÇOK ÖNEMLİ)
    news_df = news_df.rename(columns={"label": "x", "score": "y"})
    sentiment_df = sentiment_df.rename(columns={"label": "x", "score": "y"})

    # en iyi sonuç
    best_news = news_df.sort_values("y", ascending=False).iloc[0]
    best_sent = sentiment_df.sort_values("y", ascending=False).iloc[0]

    summary = f"""
### ✅ Analiz Sonucu

📰 **Kategori:** {best_news['x']} (%{best_news['y']:.2f})
😊 **Duygu:** {best_sent['x']} (%{best_sent['y']:.2f})
"""

    return summary, news_df, sentiment_df


examples = [
    ["Galatasaray derbiyi kazandı ve taraftarlar büyük sevinç yaşadı"],
    ["Merkez Bankası faiz oranlarını artırma kararı aldı"],
    ["Apple yeni yapay zeka destekli telefonunu tanıttı"],
    ["Bu ürün harika çok memnun kaldım"],
    ["Hiç memnun kalmadım ürün çok kötüydü"],
    ["Toplantı yarın saat 10:00'da yapılacak"]
]


with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🇹🇷 Türkçe Haber ve Duygu Analiz Uygulaması")
    gr.Markdown("Metni gir → kategori + duygu + grafik sonucu gör 🚀")

    text_input = gr.Textbox(
        label="📝 Metin Gir",
        placeholder="Örnek: Galatasaray derbiyi kazandı...",
        lines=5
    )

    btn = gr.Button("🚀 Analiz Et", variant="primary")

    summary_output = gr.Markdown()

    gr.Markdown("## 📊 Güven Skoru Grafikleri")

    with gr.Row():
        news_plot = gr.BarPlot(x="x", y="y", title="📰 Kategori Güvenleri")
        sentiment_plot = gr.BarPlot(x="x", y="y", title="😊 Duygu Güvenleri")

    gr.Markdown("## 🧪 Hazır Örnekler")

    gr.Examples(examples=examples, inputs=text_input)

    btn.click(
        fn=predict_both_pretty,
        inputs=text_input,
        outputs=[summary_output, news_plot, sentiment_plot]
    )

demo.launch(share=True)

/tmp/ipykernel_5447/3902988799.py:43: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://086f925296803f8733.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import pandas as pd
import matplotlib.pyplot as plt

def make_bar_chart(results, title):
    labels = [item["label"] for item in results]
    scores = [item["score"] * 100 for item in results]

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(labels, scores)
    ax.set_xlabel("Güven Skoru (%)")
    ax.set_title(title)
    ax.set_xlim(0, 100)

    for i, v in enumerate(scores):
        ax.text(v + 1, i, f"%{v:.1f}", va="center")

    plt.tight_layout()
    return fig

def predict_both_pretty(text):
    news_result = news_classifier(text, top_k=None)
    sentiment_result = sentiment_classifier(text, top_k=None)

    best_news = max(news_result, key=lambda x: x["score"])
    best_sentiment = max(sentiment_result, key=lambda x: x["score"])

    news_score = best_news["score"] * 100
    sentiment_score = best_sentiment["score"] * 100

    summary = f"""
## ✅ Analiz Sonucu

### 📰 Haber Kategorisi
**{best_news["label"]}**
Güven Skoru: **%{news_score:.2f}**

### 😊 Duygu Analizi
**{best_sentiment["label"]}**
Güven Skoru: **%{sentiment_score:.2f}**
"""

    news_chart = make_bar_chart(news_result, "Haber Kategorisi Güven Skorları")
    sentiment_chart = make_bar_chart(sentiment_result, "Duygu Analizi Güven Skorları")

    return summary, news_chart, sentiment_chart

examples = [
    ["Galatasaray derbiyi kazandı ve taraftarlar büyük sevinç yaşadı"],
    ["Merkez Bankası faiz oranlarını artırma kararı aldı"],
    ["Apple yeni yapay zeka destekli telefonunu tanıttı"],
    ["Bu ürün harika çok memnun kaldım"],
    ["Hiç memnun kalmadım ürün çok kötüydü"],
    ["Toplantı yarın saat 10:00'da yapılacak"]
]

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🇹🇷 Türkçe Haber ve Duygu Analiz Uygulaması")
    gr.Markdown("BERTurk modeli ile metnin haber kategorisi ve duygu durumu tahmin edilir.")

    with gr.Row():
        with gr.Column(scale=2):
            text_input = gr.Textbox(
                label="📝 Metin Gir",
                placeholder="Örnek: Galatasaray derbiyi kazandı...",
                lines=7
            )

            analyze_btn = gr.Button("🚀 Analiz Et", variant="primary")

            gr.Markdown("## 🧪 Hazır Örnekler")
            gr.Examples(examples=examples, inputs=text_input)

        with gr.Column(scale=1):
            summary_output = gr.Markdown(label="Analiz Sonucu")

    gr.Markdown("## 📊 Güven Skoru Grafikleri")

    with gr.Row():
        news_plot = gr.Plot(label="Haber Kategorisi Grafiği")
        sentiment_plot = gr.Plot(label="Duygu Analizi Grafiği")

    analyze_btn.click(
        fn=predict_both_pretty,
        inputs=text_input,
        outputs=[summary_output, news_plot, sentiment_plot]
    )

demo.launch(share=True)

/tmp/ipykernel_5447/1593168419.py:57: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d9a2f8e63c072d0966.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
!pip install -q transformers gradio matplotlib pandas

In [12]:
from transformers import pipeline

news_model_path = "/content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/news_model"
sentiment_model_path = "/content/drive/MyDrive/CIF425_Turkce_NLP_Projesi/sentiment_model"

news_classifier = pipeline(
    "text-classification",
    model=news_model_path,
    tokenizer=news_model_path
)

sentiment_classifier = pipeline(
    "text-classification",
    model=sentiment_model_path,
    tokenizer=sentiment_model_path
)

print("Modeller yüklendi.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modeller yüklendi.


In [13]:
import gradio as gr
import matplotlib.pyplot as plt

def make_bar_chart(results, title):
    labels = [item["label"] for item in results]
    scores = [item["score"] * 100 for item in results]

    fig, ax = plt.subplots(figsize=(6, 3))

    ax.barh(labels, scores)

    ax.set_xlabel("Güven Skoru (%)")
    ax.set_title(title)
    ax.set_xlim(0, 100)

    for i, v in enumerate(scores):
        ax.text(v + 1, i, f"%{v:.1f}", va="center")

    plt.tight_layout()

    return fig


def predict_both_pretty(text):

    news_result = news_classifier(text, top_k=None)
    sentiment_result = sentiment_classifier(text, top_k=None)

    best_news = max(news_result, key=lambda x: x["score"])
    best_sentiment = max(sentiment_result, key=lambda x: x["score"])

    news_score = best_news["score"] * 100
    sentiment_score = best_sentiment["score"] * 100

    summary = f"""
## ✅ Analiz Sonucu

### 📰 Haber Kategorisi
**{best_news["label"]}**
Güven Skoru: **%{news_score:.2f}**

### 😊 Duygu Analizi
**{best_sentiment["label"]}**
Güven Skoru: **%{sentiment_score:.2f}**
"""

    news_chart = make_bar_chart(
        news_result,
        "Haber Kategorisi Güven Skorları"
    )

    sentiment_chart = make_bar_chart(
        sentiment_result,
        "Duygu Analizi Güven Skorları"
    )

    return summary, news_chart, sentiment_chart


examples = [
    ["Galatasaray derbiyi kazandı ve taraftarlar büyük sevinç yaşadı"],
    ["Merkez Bankası faiz oranlarını artırma kararı aldı"],
    ["Apple yeni yapay zeka destekli telefonunu tanıttı"],
    ["Bu ürün harika çok memnun kaldım"],
    ["Hiç memnun kalmadım ürün çok kötüydü"],
    ["Toplantı yarın saat 10:00'da yapılacak"]
]


with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🇹🇷 Türkçe Haber ve Duygu Analiz Uygulaması")

    gr.Markdown(
        "BERTurk modeli ile metnin haber kategorisi ve duygu durumu tahmin edilir."
    )

    with gr.Row():

        with gr.Column(scale=2):

            text_input = gr.Textbox(
                label="📝 Metin Gir",
                placeholder="Örnek: Galatasaray derbiyi kazandı...",
                lines=7
            )

            analyze_btn = gr.Button(
                "🚀 Analiz Et",
                variant="primary"
            )

            gr.Markdown("## 🧪 Hazır Örnekler")

            gr.Examples(
                examples=examples,
                inputs=text_input
            )

        with gr.Column(scale=1):

            summary_output = gr.Markdown(
                label="Analiz Sonucu"
            )

    gr.Markdown("## 📊 Güven Skoru Grafikleri")

    with gr.Row():

        news_plot = gr.Plot(
            label="Haber Kategorisi Grafiği"
        )

        sentiment_plot = gr.Plot(
            label="Duygu Analizi Grafiği"
        )

    analyze_btn.click(
        fn=predict_both_pretty,
        inputs=text_input,
        outputs=[
            summary_output,
            news_plot,
            sentiment_plot
        ]
    )

demo.launch(share=True)

/tmp/ipykernel_5543/66502808.py:70: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a27b0633aabebbed41.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
